# 02 — Fraud Pattern Visualization

This notebook visualizes the four injected fraud patterns:
1. **Mule Rings** — Device-sharing subgraphs
2. **Burst Attacks** — Velocity spikes in short time windows
3. **Merchant Collusion** — Benford's Law deviation from round-number amounts
4. **Account Takeover** — Impossible geographic velocity

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from datetime import timedelta

from data.synthetic.generator import SyntheticUPIGenerator
from data.features.benford import benford_expected_distribution, first_digit_frequencies

sns.set_theme(style='whitegrid')
%matplotlib inline

In [ ]:
gen = SyntheticUPIGenerator(n_users=5000, n_merchants=500, n_transactions=25000, fraud_ratio=0.05, seed=42)
df = gen.generate()
fraud_df = df[df['is_fraud']].copy()
normal_df = df[~df['is_fraud']].copy()

print(f'Total: {len(df)} | Fraud: {len(fraud_df)} ({len(fraud_df)/len(df)*100:.1f}%)')
print(f'\nFraud pattern counts:')
print(fraud_df['fraud_pattern'].value_counts())

## 1. Mule Rings — Shared Device Fingerprints

Mule ring members share device fingerprints. This creates dense subgraph connectivity.

In [ ]:
mule_df = fraud_df[fraud_df['fraud_pattern'] == 'MULE_RING']
print(f'Mule ring transactions: {len(mule_df)}')

shared_devices = mule_df['device_fingerprint'].value_counts()
shared_devices = shared_devices[shared_devices > 1]
print(f'\nShared devices (used by >1 account):')
for dev, count in shared_devices.items():
    users = mule_df[mule_df['device_fingerprint'] == dev]['user_id'].unique()
    print(f'  {dev}: shared by {len(users)} users -> {list(users[:5])}')

In [ ]:
G = nx.MultiDiGraph()
for _, row in mule_df.iterrows():
    G.add_node(row['user_id'], node_type='user')
    G.add_node(row['device_fingerprint'], node_type='device')
    G.add_node(row['merchant_id'], node_type='merchant')
    G.add_edge(row['user_id'], row['device_fingerprint'], edge_type='used')
    G.add_edge(row['user_id'], row['merchant_id'], edge_type='paid_to')

plt.figure(figsize=(14, 10))
pos = nx.spring_layout(G, k=2, iterations=50)
user_nodes = [n for n, d in G.nodes(data=True) if d.get('node_type') == 'user']
device_nodes = [n for n, d in G.nodes(data=True) if d.get('node_type') == 'device']
merchant_nodes = [n for n, d in G.nodes(data=True) if d.get('node_type') == 'merchant']

nx.draw_networkx_nodes(G, pos, nodelist=user_nodes, node_color='red', node_size=100, label='Users')
nx.draw_networkx_nodes(G, pos, nodelist=device_nodes, node_color='blue', node_size=50, label='Devices')
nx.draw_networkx_nodes(G, pos, nodelist=merchant_nodes, node_color='green', node_size=80, label='Merchants')
nx.draw_networkx_edges(G, pos, alpha=0.3, arrows=False)
plt.title('Mule Ring Subgraph: Shared Device Fingerprints')
plt.legend()
plt.axis('off')
plt.show()

## 2. Burst Attacks — Transaction Velocity

Burst attacks show a sudden spike of 15-30 transactions within 5 minutes.

In [ ]:
burst_df = fraud_df[fraud_df['fraud_pattern'] == 'BURST_ATTACK'].copy()
burst_df = burst_df.sort_values('timestamp')

if len(burst_df) > 0:
    burst_df['minute_group'] = burst_df['timestamp'].dt.floor('1min')
    velocity = burst_df.groupby(['user_id', 'minute_group']).size().reset_index(name='txn_count')

    top_bursts = velocity.nlargest(10, 'txn_count')
    print('Top burst events:')
    print(top_bursts)

    plt.figure(figsize=(12, 5))
    for uid in burst_df['user_id'].unique()[:3]:
        user_data = burst_df[burst_df['user_id'] == uid].sort_values('timestamp')
        plt.plot(user_data['timestamp'], range(len(user_data)), marker='o', label=f'User {uid[:8]}')
    plt.xlabel('Timestamp')
    plt.ylabel('Cumulative Transactions')
    plt.title('Burst Attack: Transaction Accumulation Over Time')
    plt.legend()
    plt.show()

## 3. Merchant Collusion — Benford's Law Deviation

Shell merchants use round-number amounts, causing first-digit distribution to deviate from Benford's Law.

In [ ]:
collusion_df = fraud_df[fraud_df['fraud_pattern'] == 'MERCHANT_COLLUSION']
shell_merchants = collusion_df['merchant_id'].unique()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Shell merchants first digits
for i, mid in enumerate(shell_merchants[:3]):
    amounts = collusion_df[collusion_df['merchant_id'] == mid]['amount'].tolist()
    if not amounts:
        continue
    observed = first_digit_frequencies(amounts)
    expected = benford_expected_distribution()
    
    axes[i].bar(np.arange(1, 10) - 0.2, expected, width=0.3, label='Expected (Benford)', alpha=0.7, color='gray')
    axes[i].bar(np.arange(1, 10) + 0.2, observed, width=0.3, label=f'Observed ({mid})', alpha=0.7, color='red')
    axes[i].set_title(f'Merchant {mid}')
    axes[i].set_xlabel('First Digit')
    axes[i].legend()

plt.suptitle('Benford Law Deviation: Shell Merchants vs Expected')
plt.tight_layout()
plt.show()

# Also show a normal merchant for comparison
normal_amounts = normal_df['amount'].sample(1000).tolist()
normal_observed = first_digit_frequencies(normal_amounts)
expected = benford_expected_distribution()

plt.figure(figsize=(8, 5))
plt.bar(np.arange(1, 10) - 0.2, expected, width=0.3, label='Expected (Benford)', alpha=0.7, color='gray')
plt.bar(np.arange(1, 10) + 0.2, normal_observed, width=0.3, label='Observed (Normal)', alpha=0.7, color='green')
plt.title('Benford Law: Normal Merchant (Expected Fit)')
plt.xlabel('First Digit')
plt.legend()
plt.show()

## 4. Account Takeover — Impossible Geographic Velocity

ATO transactions show location jumps > 500km within minutes, which is physically impossible.

In [ ]:
ato_df = fraud_df[fraud_df['fraud_pattern'] == 'ATO'].copy()

if len(ato_df) > 0:
    plt.figure(figsize=(12, 6))
    
    for uid in ato_df['user_id'].unique()[:3]:
        user_data = ato_df[ato_df['user_id'] == uid].sort_values('timestamp')
        plt.scatter(user_data['lon'], user_data['lat'], s=100, label=f'User {uid[:8]} (fraud)', marker='x')
        for _, row in user_data.iterrows():
            plt.annotate(row['timestamp'].strftime('%H:%M'), (row['lon'], row['lat']), fontsize=8)
    
    # Plot normal user locations for reference
    normal_users = normal_df.groupby('user_id').first().reset_index()
    plt.scatter(normal_users['lon'], normal_users['lat'], alpha=0.3, s=5, c='gray', label='Normal users')
    
    plt.xlabel('Longitude')
    plt.ylabel('Latitude')
    plt.title('Account Takeover: Impossible Geographic Location Jumps')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()
    
print(f'ATO transactions: {len(ato_df)}')
print(f'Unique compromised users: {ato_df["user_id"].nunique()}')

## Summary

All four fraud patterns are visually distinguishable from normal behavior:
- **Mule Rings**: Dense shared-device subgraphs
- **Burst Attacks**: Steep transaction accumulation within minutes
- **Merchant Collusion**: First-digit distributions that violate Benford's Law
- **Account Takeover**: Geographic coordinates that imply impossible travel velocity